In [2]:
# Required Libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Load datasets
sdn_ddos = pd.read_csv('/content/SDN_DDOS_dataset.csv')
icmp = pd.read_csv('/content/ICMP.csv')
udp = pd.read_csv('/content/udp.csv')

# Drop unnecessary column from SDN_DDOS if it exists
if 'Unnamed: 0' in sdn_ddos.columns:
    sdn_ddos.drop(columns=['Unnamed: 0'], inplace=True)

# Combine datasets by concatenation (assuming they are separate samples with the same features)
data = pd.concat([sdn_ddos, icmp, udp], ignore_index=True)

# Display data shape after concatenation
print("Data shape after concatenation:", data.shape)

# Check for NaNs in 'label'
missing_labels = data['label'].isna().sum()
print(f"Number of samples with missing 'label': {missing_labels}")

# Drop rows with missing 'label'
data = data.dropna(subset=['label'])

# Verify that 'label' has no NaNs
assert data['label'].isna().sum() == 0, "There are still NaNs in the 'label' column."

# Features and Labels split
X = data.drop('label', axis=1)
y = data['label']

# Select only numeric columns for feature set
X_numeric = X.select_dtypes(include=[np.number])

# Ensure X_numeric contains data
if X_numeric.empty:
    raise ValueError("No numeric columns or data found in X_numeric. Check dataset structure.")

# Check for NaNs in y (redundant after dropping, but safe to verify)
missing_y = y.isna().sum()
if missing_y > 0:
    print(f"Number of samples with missing y: {missing_y}")
    # Drop these rows if any (shouldn't be necessary)
    data = data.dropna(subset=['label'])
    X_numeric = X_numeric.loc[data.index]
    y = y.loc[data.index]

# Now check again
if X_numeric.empty or y.empty:
    raise ValueError("No data available after handling missing values.")

# Split dataset into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_numeric, y, test_size=0.3, random_state=42, stratify=y
)

# Add Gaussian noise to the dataset
noise_factor = 0.2
X_train_noisy = X_train + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=X_train.shape)
X_test_noisy = X_test + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=X_test.shape)

# Clip to ensure data stays in valid range (useful if your data is bounded)
X_train_noisy = np.clip(X_train_noisy, 0., 1.)
X_test_noisy = np.clip(X_test_noisy, 0., 1.)

# Handle missing values using SimpleImputer
imputer = SimpleImputer(strategy='mean')
X_train_noisy = imputer.fit_transform(X_train_noisy)
X_test_noisy = imputer.transform(X_test_noisy)

# Scale the data using StandardScaler
scaler = StandardScaler()
X_train_noisy = scaler.fit_transform(X_train_noisy)
X_test_noisy = scaler.transform(X_test_noisy)

# Define a function to calculate and print evaluation metrics
def evaluate_model(model, X_test, y_test, model_name="Model"):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    print(f"\n{model_name} Metrics:")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(classification_report(y_test, y_pred, zero_division=0))

# 1. K-Nearest Neighbors (KNN) model
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train_noisy, y_train)
evaluate_model(knn, X_test_noisy, y_test, "K-Nearest Neighbors")

# 2. Naive Bayes model
nb = GaussianNB()
nb.fit(X_train_noisy, y_train)
evaluate_model(nb, X_test_noisy, y_test, "Naive Bayes")

# 3. Decision Tree model
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train_noisy, y_train)
evaluate_model(dt, X_test_noisy, y_test, "Decision Tree")

# 4. Support Vector Machine (SVM)
svm = SVC(kernel='linear', random_state=42)
svm.fit(X_train_noisy, y_train)
evaluate_model(svm, X_test_noisy, y_test, "Support Vector Machine")

# 5. Random Forest model with adjusted hyperparameters to reduce overfitting
rf = RandomForestClassifier(
    n_estimators=40,            # Fewer trees
    max_depth=4,               # Limit tree depth
    min_samples_split=8,       # Minimum samples required to split an internal node
    min_samples_leaf=5,         # Minimum samples required to be at a leaf node
    random_state=42
)
rf.fit(X_train_noisy, y_train)
evaluate_model(rf, X_test_noisy, y_test, "Random Forest")

# 6. Logistic Regression
lr = LogisticRegression(max_iter=100, random_state=42)
lr.fit(X_train_noisy, y_train)
evaluate_model(lr, X_test_noisy, y_test, "Logistic Regression")

# 7. Ensemble Model (Voting Classifier)
ensemble = VotingClassifier(estimators=[
    ('knn', knn),
    ('nb', nb),
    ('dt', dt),
    ('svm', svm),
    ('rf', rf),  # Updated Random Forest
    ('lr', lr)
], voting='hard')

ensemble.fit(X_train_noisy, y_train)
evaluate_model(ensemble, X_test_noisy, y_test, "Ensemble Voting Classifier")

# Cross-validation to check generalization performance
cv = StratifiedKFold(n_splits=5)
print("\nCross-validated KNN accuracy:", np.mean(cross_val_score(knn, X_train_noisy, y_train, cv=cv)))
print("Cross-validated Naive Bayes accuracy:", np.mean(cross_val_score(nb, X_train_noisy, y_train, cv=cv)))
print("Cross-validated Decision Tree accuracy:", np.mean(cross_val_score(dt, X_train_noisy, y_train, cv=cv)))
print("Cross-validated SVM accuracy:", np.mean(cross_val_score(svm, X_train_noisy, y_train, cv=cv)))
print("Cross-validated Random Forest accuracy:", np.mean(cross_val_score(rf, X_train_noisy, y_train, cv=cv)))
print("Cross-validated Logistic Regression accuracy:", np.mean(cross_val_score(lr, X_train_noisy, y_train, cv=cv)))
print("Cross-validated Ensemble accuracy:", np.mean(cross_val_score(ensemble, X_train_noisy, y_train, cv=cv)))


Data shape after concatenation: (119098, 15)
Number of samples with missing 'label': 0

K-Nearest Neighbors Metrics:
Accuracy: 0.9386
Precision: 0.9384
Recall: 0.9386
F1 Score: 0.9385
              precision    recall  f1-score   support

           0       0.96      0.96      0.96     26467
           1       0.89      0.87      0.88      9263

    accuracy                           0.94     35730
   macro avg       0.92      0.92      0.92     35730
weighted avg       0.94      0.94      0.94     35730


Naive Bayes Metrics:
Accuracy: 0.9043
Precision: 0.9198
Recall: 0.9043
F1 Score: 0.9075
              precision    recall  f1-score   support

           0       0.98      0.89      0.93     26467
           1       0.75      0.95      0.84      9263

    accuracy                           0.90     35730
   macro avg       0.86      0.92      0.88     35730
weighted avg       0.92      0.90      0.91     35730


Decision Tree Metrics:
Accuracy: 0.9231
Precision: 0.9232
Recall: 0.9231